# 벤치마크: Vector RAG vs GraphRAG (20문항)

**소요 시간**: ~30분 (API 호출 포함)  
**난이도**: 중급  

---

## 개요

이 노트북은 **Vector RAG**와 **GraphRAG (Text2Cypher)** 두 가지 방식을 20문항 벤치마크로 비교합니다.  
RAGAS 평가 프레임워크를 사용하여 faithfulness, answer_relevancy, context_precision, context_recall 4개 메트릭으로 정량 평가합니다.

### 사전 조건

| # | 조건 | 설명 |
|---|------|------|
| 1 | **Part 2 노트북 실행 완료** | Neo4j에 제조 도메인 KG가 로드되어 있어야 합니다 |
| 2 | **Docker Neo4j 실행 중** | `bolt://localhost:7687` 접속 가능 |
| 3 | **`.env` 설정** | `OPENAI_API_KEY`, `NEO4J_PASSWORD` 설정 필요 |

### 목차

0. 환경 설정
1. Knowledge Graph 상태 확인
2. 평가 데이터셋 로드 (20문항)
3. Vector RAG Baseline
4. GraphRAG -- Text2Cypher
5. RAGAS 평가
6. 결과 분석 및 시각화
7. 결과 내보내기

In [ ]:
import os
import json
import time
from collections import Counter

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase
from IPython.display import display

In [ ]:
# 환경 변수 로드
load_dotenv()
load_dotenv(dotenv_path="../.env")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USERNAME", os.getenv("NEO4J_USER", "neo4j"))
NEO4J_PASS = os.getenv("NEO4J_PASSWORD", "")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

# Neo4j 연결
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

try:
    driver.verify_connectivity()
    print(f"[OK] Neo4j 연결 성공: {NEO4J_URI}")
except Exception as e:
    print(f"[ERROR] Neo4j 연결 실패: {e}")
    print("Neo4j가 실행 중인지, .env 설정이 올바른지 확인하세요.")

# 편의 함수
def run_query(query, params=None):
    """Cypher 쿼리를 실행하고 결과를 반환합니다."""
    with driver.session() as session:
        result = session.run(query, params or {})
        return [record.data() for record in result]

print(f"OpenAI API Key: {'설정됨' if OPENAI_API_KEY else '미설정 ⚠️'}")
print("[OK] run_query 함수 준비 완료")

---
## 1. Knowledge Graph 상태 확인

Part 2 노트북을 먼저 실행하여 KG를 로드해야 합니다.  
아래 셀에서 노드/관계 수를 확인하고, KG가 비어있으면 오류를 발생시킵니다.

In [ ]:
stats = run_query("""
MATCH (n) WITH labels(n) AS labels, count(*) AS cnt
UNWIND labels AS label
RETURN label, sum(cnt) AS count ORDER BY count DESC
""")
print("=== 노드 통계 ===")
for s in stats:
    print(f"  {s['label']}: {s['count']}개")

rel_stats = run_query("""
MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS count ORDER BY count DESC
""")
print("\n=== 관계 통계 ===")
for s in rel_stats:
    print(f"  {s['type']}: {s['count']}개")

total_nodes = sum(s['count'] for s in stats)
total_rels = sum(s['count'] for s in rel_stats)
assert total_nodes > 0, "⚠️ KG가 비어있습니다. Part 2 노트북을 먼저 실행하세요."
print(f"\n✅ KG 로드 확인: {total_nodes}개 노드, {total_rels}개 관계")

---
## 2. 평가 데이터셋 로드 (20문항)

`data/eval_questions.json`에서 난이도별(easy/medium/hard) 20문항을 로드합니다.  
각 문항에는 질문, 정답(ground_truth_answer), 난이도, 예상 hop 수가 포함되어 있습니다.

In [ ]:
with open("data/eval_questions.json", "r", encoding="utf-8") as f:
    eval_questions = json.load(f)

print(f"총 {len(eval_questions)}문항 로드")

difficulty_dist = Counter(q["difficulty"] for q in eval_questions)
for diff, cnt in sorted(difficulty_dist.items()):
    print(f"  {diff}: {cnt}문항")

# 미리보기
df = pd.DataFrame(eval_questions)[["id", "question", "difficulty", "expected_hops"]]
display(df)

---
## 3. Vector RAG Baseline

`manufacturing_docs.json` 텍스트를 FAISS에 임베딩하여 Vector RAG baseline을 구축합니다.  
top-k=3으로 검색한 문서 컨텍스트를 GPT-4o-mini에 전달하여 답변을 생성합니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.schema import Document

# 제조 문서 로드
with open("data/manufacturing_docs.json", "r", encoding="utf-8") as f:
    docs_raw = json.load(f)

documents = [
    Document(
        page_content=d["content"],
        metadata={"title": d["title"], "id": d["id"]}
    )
    for d in docs_raw
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"✅ FAISS 인덱스 생성: {len(documents)}개 문서")

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

vector_results = []
for i, q in enumerate(eval_questions):
    # 관련 문서 검색
    retrieved = retriever.invoke(q["question"])
    context = "\n\n".join([doc.page_content for doc in retrieved])

    # 답변 생성
    prompt = f"""다음 컨텍스트를 기반으로 질문에 답하세요. 컨텍스트에 없는 정보는 "정보 없음"이라고 답하세요.

컨텍스트:
{context}

질문: {q['question']}
답변:"""

    response = llm.invoke(prompt)

    vector_results.append({
        "id": q["id"],
        "question": q["question"],
        "difficulty": q["difficulty"],
        "ground_truth": q["ground_truth_answer"],
        "vector_answer": response.content,
        "vector_contexts": [doc.page_content for doc in retrieved]
    })
    print(f"  [{i+1}/20] {q['id']}: {response.content[:60]}...")

print(f"\n✅ Vector RAG 완료: {len(vector_results)}문항")

---
## 4. GraphRAG -- Text2Cypher

Part 6에서 학습한 Text2Cypher 방식으로 동일 질문에 답변합니다.  
LLM이 자연어 질문을 Cypher 쿼리로 변환하고, Neo4j에서 실행한 결과를 기반으로 답변을 생성합니다.

In [ ]:
# KG 스키마 정보 조회
node_labels = run_query(
    "CALL db.labels() YIELD label RETURN collect(label) AS labels"
)[0]["labels"]
rel_types = run_query(
    "CALL db.relationshipTypes() YIELD relationshipType RETURN collect(relationshipType) AS types"
)[0]["types"]

schema_text = f"""Node Labels: {', '.join(node_labels)}
Relationship Types: {', '.join(rel_types)}

주요 패턴:
- (Process)-[:NEXT_PROCESS]->(Process)
- (Process)-[:USES_EQUIPMENT]->(Equipment)
- (Process)-[:USES_MATERIAL]->(Material)
- (Defect)-[:CAUSED_BY]->(Equipment)
- (Defect)-[:DETECTED_IN]->(Process)
- (Maintenance)-[:MAINTAINED_ON]->(Equipment)
- (Process)-[:CONFORMS_TO]->(Spec)
- (Process)-[:PRODUCES]->(Product)
- (Product)-[:INSPECTED_BY]->(Equipment)"""

print("KG 스키마:")
print(schema_text)

In [ ]:
graph_results = []

for i, q in enumerate(eval_questions):
    # Step 1: Cypher 쿼리 생성
    cypher_prompt = f"""당신은 Neo4j Cypher 쿼리 전문가입니다. 아래 스키마를 기반으로 질문에 답하는 Cypher 쿼리를 생성하세요.

스키마:
{schema_text}

질문: {q['question']}

Cypher 쿼리만 출력하세요 (설명 불필요):"""

    cypher_response = llm.invoke(cypher_prompt)
    generated_cypher = (
        cypher_response.content.strip()
        .replace("```cypher", "")
        .replace("```", "")
        .strip()
    )

    # Step 2: Cypher 실행
    try:
        cypher_result = run_query(generated_cypher)
        context = json.dumps(cypher_result, ensure_ascii=False, default=str)
    except Exception as e:
        context = f"쿼리 실행 오류: {str(e)}"

    # Step 3: 쿼리 결과 기반 답변 생성
    answer_prompt = f"""다음은 Neo4j 그래프 데이터베이스 쿼리 결과입니다. 이를 기반으로 질문에 자연어로 답하세요.

질문: {q['question']}
쿼리 결과: {context}

답변:"""

    answer_response = llm.invoke(answer_prompt)

    graph_results.append({
        "id": q["id"],
        "question": q["question"],
        "difficulty": q["difficulty"],
        "ground_truth": q["ground_truth_answer"],
        "generated_cypher": generated_cypher,
        "cypher_result": context,
        "graph_answer": answer_response.content,
        "graph_contexts": [context]
    })
    print(f"  [{i+1}/20] {q['id']}: {answer_response.content[:60]}...")

print(f"\n✅ GraphRAG 완료: {len(graph_results)}문항")

---
## 5. RAGAS 평가

RAGAS 프레임워크의 4개 메트릭으로 두 방식을 정량 비교합니다.

| 메트릭 | 설명 |
|--------|------|
| **faithfulness** | 답변이 컨텍스트에 충실한가? |
| **answer_relevancy** | 답변이 질문에 관련 있는가? |
| **context_precision** | 검색된 컨텍스트가 정확한가? |
| **context_recall** | 필요한 컨텍스트를 모두 검색했는가? |

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

# Vector RAG 데이터셋 준비
vector_dataset = Dataset.from_dict({
    "question": [r["question"] for r in vector_results],
    "answer": [r["vector_answer"] for r in vector_results],
    "contexts": [r["vector_contexts"] for r in vector_results],
    "ground_truth": [r["ground_truth"] for r in vector_results],
})

# GraphRAG 데이터셋 준비
graph_dataset = Dataset.from_dict({
    "question": [r["question"] for r in graph_results],
    "answer": [r["graph_answer"] for r in graph_results],
    "contexts": [r["graph_contexts"] for r in graph_results],
    "ground_truth": [r["ground_truth"] for r in graph_results],
})

print("Vector RAG RAGAS 평가 중...")
vector_scores = evaluate(
    vector_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
)

print("GraphRAG RAGAS 평가 중...")
graph_scores = evaluate(
    graph_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
)

print("\n=== Vector RAG ===")
print(vector_scores)
print("\n=== GraphRAG ===")
print(graph_scores)

In [ ]:
# 난이도별 분석
difficulties = ["easy", "medium", "hard"]

results_by_difficulty = {}
for diff in difficulties:
    v_indices = [i for i, r in enumerate(vector_results) if r["difficulty"] == diff]
    g_indices = [i for i, r in enumerate(graph_results) if r["difficulty"] == diff]

    # 간단한 정확도: ground_truth 키워드 포함 여부
    v_correct = 0
    g_correct = 0
    for idx in v_indices:
        gt_keywords = vector_results[idx]["ground_truth"].split()[:3]
        if any(kw in vector_results[idx]["vector_answer"] for kw in gt_keywords if len(kw) > 1):
            v_correct += 1
    for idx in g_indices:
        gt_keywords = graph_results[idx]["ground_truth"].split()[:3]
        if any(kw in graph_results[idx]["graph_answer"] for kw in gt_keywords if len(kw) > 1):
            g_correct += 1

    total = len(v_indices)
    results_by_difficulty[diff] = {
        "total": total,
        "vector_correct": v_correct,
        "vector_accuracy": round(v_correct / total * 100, 1) if total > 0 else 0,
        "graph_correct": g_correct,
        "graph_accuracy": round(g_correct / total * 100, 1) if total > 0 else 0,
    }

# 테이블 출력
rows = []
for diff in difficulties:
    r = results_by_difficulty[diff]
    rows.append({
        "난이도": diff.capitalize(),
        "문항수": r["total"],
        "Vector RAG": f"{r['vector_accuracy']}% ({r['vector_correct']}/{r['total']})",
        "GraphRAG": f"{r['graph_accuracy']}% ({r['graph_correct']}/{r['total']})",
        "차이": f"+{r['graph_accuracy'] - r['vector_accuracy']}%p"
    })

df_results = pd.DataFrame(rows)
display(df_results)

---
## 6. 결과 분석 및 시각화

In [ ]:
print("=" * 60)
print("벤치마크 결과 요약 -- Vector RAG vs GraphRAG (Text2Cypher)")
print("=" * 60)
print(f"\n평가 모델: gpt-4o-mini | 질문 수: 20 | KG: 제조 도메인")
print(f"\n{'난이도':<12} {'Vector RAG':<15} {'GraphRAG':<15} {'차이':<10}")
print("-" * 52)
for diff in difficulties:
    r = results_by_difficulty[diff]
    delta = r['graph_accuracy'] - r['vector_accuracy']
    sign = "+" if delta >= 0 else ""
    print(f"{diff.capitalize():<12} {r['vector_accuracy']:>6.1f}%        {r['graph_accuracy']:>6.1f}%        {sign}{delta:.1f}%p")

# 전체 정확도
total_v = sum(r["vector_correct"] for r in results_by_difficulty.values())
total_g = sum(r["graph_correct"] for r in results_by_difficulty.values())
total_q = sum(r["total"] for r in results_by_difficulty.values())
v_overall = round(total_v / total_q * 100, 1)
g_overall = round(total_g / total_q * 100, 1)
print("-" * 52)
print(f"{'전체':<12} {v_overall:>6.1f}%        {g_overall:>6.1f}%        +{g_overall - v_overall:.1f}%p")

print(f"\nRAGAS 점수:")
print(f"  Vector RAG: {vector_scores}")
print(f"  GraphRAG:   {graph_scores}")

In [ ]:
print("\n=== 문항별 상세 결과 ===\n")
for v, g in zip(vector_results, graph_results):
    gt_keywords = v["ground_truth"].split()[:3]
    v_match = "✅" if any(kw in v["vector_answer"] for kw in gt_keywords if len(kw) > 1) else "❌"
    g_match = "✅" if any(kw in g["graph_answer"] for kw in gt_keywords if len(kw) > 1) else "❌"
    print(f"[{v['id']}] ({v['difficulty']}) {v['question']}")
    print(f"  정답: {v['ground_truth'][:80]}")
    print(f"  Vector: {v_match} {v['vector_answer'][:80]}")
    print(f"  Graph:  {g_match} {g['graph_answer'][:80]}")
    print()

---
## 7. 결과 내보내기

벤치마크 결과를 JSON 파일로 저장합니다.  
이 파일을 사용하여 커리큘럼 콘텐츠의 예시 수치를 실측값으로 교체할 수 있습니다.

In [ ]:
benchmark_output = {
    "metadata": {
        "date": time.strftime("%Y-%m-%d %H:%M"),
        "model": "gpt-4o-mini",
        "num_questions": 20,
        "domain": "제조 (브레이크패드)",
        "kg_source": "manufacturing_docs.json (5 documents)"
    },
    "ragas_scores": {
        "vector_rag": {k: float(v) for k, v in vector_scores.items()},
        "graphrag": {k: float(v) for k, v in graph_scores.items()}
    },
    "accuracy_by_difficulty": results_by_difficulty,
    "overall": {
        "vector_rag_accuracy": v_overall,
        "graphrag_accuracy": g_overall,
        "delta": round(g_overall - v_overall, 1)
    },
    "per_question": [
        {
            "id": v["id"],
            "difficulty": v["difficulty"],
            "question": v["question"],
            "ground_truth": v["ground_truth"],
            "vector_answer": v["vector_answer"],
            "graph_answer": g["graph_answer"],
            "generated_cypher": g["generated_cypher"]
        }
        for v, g in zip(vector_results, graph_results)
    ]
}

output_path = "data/benchmark_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(benchmark_output, f, ensure_ascii=False, indent=2)

print(f"✅ 벤치마크 결과 저장: {output_path}")
print(f"  이 파일을 사용하여 커리큘럼 콘텐츠의 '예시' 수치를 실측값으로 교체하세요.")

# driver.close()  # 다른 노트북에서도 사용하므로 주석 처리